In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyectofinalma")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/Movies.csv"

In [0]:
movies_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True),
    StructField("language", StringType(), True),
    StructField("user_score", DoubleType(), True),
    StructField("runtime_hour", IntegerType(), True),
    StructField("runtime_min", IntegerType(), True),
    StructField("release_date", TimestampType(), True),
    StructField("vote_count", IntegerType(), True)
])

In [0]:
races_df = spark.read \
            .option("header", True) \
            .schema(movies_schema) \
            .csv(ruta)

In [0]:
races_with_timestamp_df = races_df.withColumn("release_date", current_timestamp())

In [0]:
races_selected_df = races_with_timestamp_df.select(
    col('id').alias('movie_id'),
    col('title'),
    col('genres'),
    col('language'),
    col('user_score'),
    col('runtime_hour'),
    col('runtime_min'),
    col('release_date'),
    col('vote_count')
)

In [0]:
races_selected_df.write.mode('overwrite').saveAsTable(f'{catalogo}.{esquema}.movies')